# Course 5: Building a Small Bangla Language Model
## Session 2: Bangla text and Unicode

**Goal:** see how Bangla is stored, find look-alike forms, and write a safe cleaning function.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network.

In [ ]:
# On Colab or Kaggle, run once:  !pip install regex

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

### 1. Code points, bytes and clusters

In [ ]:
import unicodedata
import regex

def clusters(word):
    return regex.findall(r"\X", word)

for w in ["বাংলা", "ক্ষত্রিয়"]:
    print(w, "| code points:", len(w), "| bytes:", len(w.encode("utf-8")), "| clusters:", clusters(w))
for ch in "বাংলা":
    print(hex(ord(ch)), unicodedata.name(ch))

### 2. Same look, different codes

In [ ]:
a, b = "\u09df", "\u09af\u09bc"        # two ways to write the same letter
print(a, b, "equal:", a == b)
print("equal after NFC:", unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC", b))
o1, o2 = "\u0995\u09cb", "\u0995\u09c7\u09be"
print(o1, o2, "equal:", o1 == o2, "| after NFC:", unicodedata.normalize("NFC", o2) == o1)

### 3. Cleaning

In [ ]:
def clean(text):
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200b", "")              # zero-width space
    text = regex.sub(r"[ \t\u00a0]+", " ", text)
    return text.strip()

def bangla_ratio(text):
    letters = regex.findall(r"\p{L}", text)
    if not letters:
        return 0.0
    return sum(1 for c in letters if "\u0980" <= c <= "\u09ff") / len(letters)

print(repr(clean("আমি  \u200bবাংলায়\u00a0গান গাই ")))
print(bangla_ratio("আমি AI শিখি"))

### 4. Statistics of a text sample
Replace `text` with 5-10 MB of your own text.

In [ ]:
from collections import Counter
text = "\n".join(SAMPLE_LINES)          # TODO: load your own sample here
counts = Counter(text)
for ch, n in counts.most_common(15):
    print(repr(ch), n, unicodedata.name(ch, "?"))
print("precomposed U+09DF:", text.count("\u09df"), "| decomposed:", text.count("\u09af\u09bc"))
ratios = [bangla_ratio(line) for line in text.splitlines()]
print("mean Bangla ratio:", sum(ratios) / len(ratios))

### 5. TODO
- Write five test cases for `clean`, including one string that must not change.
- Plot a histogram of `ratios` for your sample and choose a threshold.
- Decide what to do with digits, the danda and English words, and write it down.